In [10]:
import sys
import os

path_to_scripts = os.path.join("..", '..', "python_scripts")
sys.path.append(path_to_scripts)

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_error, max_error

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

from data_to_bigquery import load_from_bigquery
from feature_engineering import drop_lag_nulls, validate_features
from baseline_model import baseline_model_xgb, engineer_features


%matplotlib inline

In [12]:
df = load_from_bigquery('gridzero-489711', 'merged_set', 'test_merge_2017_onward_raw')

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 148,991 rows and 26 columns from gridzero-489711.merged_set.test_merge_2017_onward_raw


In [13]:
df

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity_gCO2_kWh,status
0,2017-09-12 00:00:00,11.6,31.0,28.1,4,0.0,0.0,0.0,1001.2,0.0,...,0.0,265.0,7897.0,173.0,0.0,4006.573,4394.973,21722.546,142.0,okay
1,2017-09-12 00:30:00,11.6,31.0,28.1,4,0.0,0.0,0.0,1001.2,0.0,...,0.0,248.0,7897.0,174.0,0.0,3973.985,4418.446,21554.431,140.0,okay
2,2017-09-12 01:00:00,11.2,30.3,27.0,5,0.0,0.0,0.0,1001.9,0.0,...,0.0,242.0,7852.0,171.0,0.0,3941.698,4533.019,21767.717,139.0,okay
3,2017-09-12 01:30:00,11.2,30.3,27.0,5,0.0,0.0,0.0,1001.9,0.0,...,0.0,242.0,7706.0,174.0,0.0,3945.620,4726.191,21742.811,137.0,okay
4,2017-09-12 02:00:00,10.9,29.6,25.2,7,0.0,0.0,0.0,1002.4,0.0,...,0.0,242.0,7684.0,173.0,0.0,3919.454,4832.410,21579.864,132.0,okay
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148986,2026-03-12 21:00:00,11.4,50.8,68.0,100,0.0,0.0,0.0,1004.1,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,okay
148987,2026-03-12 21:30:00,11.4,50.8,68.0,100,0.0,0.0,0.0,1004.1,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,okay
148988,2026-03-12 22:00:00,11.4,51.5,67.3,100,0.0,0.0,0.0,1003.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,okay
148989,2026-03-12 22:30:00,11.4,51.5,67.3,100,0.0,0.0,0.0,1003.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,okay


In [8]:
def baseline_preproc(df, target_col="carbon_intensity_gCO2_kWh", add_year_lag=False):
    """
    Apply feature engineering only if engineered columns are missing.
    Drop lag-created nulls if lag columns are present.
    """

    df = df.copy()

    # rename time column if needed
    if "time" in df.columns and "datetime" not in df.columns:
        df = df.rename(columns={"time": "datetime"})

    required_features = [
        "lag_48",
        "lag_336",
        "hour",
        "day_of_week",
        "month",
        "is_weekend"
    ]

    if add_year_lag:
        required_features.append("lag_17520")

    # only engineer features if they are missing
    missing_features = [col for col in required_features if col not in df.columns]

    if missing_features:
        df = engineer_features(df, target_col=target_col, add_year_lag=add_year_lag)

    # drop nulls only if lag columns exist
    lag_cols = [col for col in ["lag_48", "lag_336", "lag_17520"] if col in df.columns]

    if lag_cols:
        df = df.dropna(subset=lag_cols).reset_index(drop=True)

    return df


In [17]:
import baseline_model
print(baseline_model.__file__)
print("baseline_preproc" in dir(baseline_model))
print(dir(baseline_model))

/Users/madeleine/code/mlbh/GridZero/notebooks/data/../../python_scripts/baseline_model.py
True
['StandardScaler', 'XGBRegressor', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'baseline_model_xgb', 'baseline_preproc', 'drop_lag_nulls', 'engineer_features', 'load_from_bigquery', 'make_pipeline', 'make_prediction', 'mean_absolute_error', 'r2_score', 'train_test_split', 'validate_features']


In [18]:
baseline_preproc(df)

NameError: name 'pd' is not defined

In [ ]:
model = baseline_model_xgb('gridzero-489711', 'merged_set', 'Fully_merged_dataset_2025')
model

In [ ]:
# def load_from_bigquery(PROJECT: str, DATASET: str, TABLE: str):
#     '''
#     Load data from BigQuery into a pandas DataFrame.

#     Arguments:
#         PROJECT: GCP project ID
#         DATASET: BigQuery dataset name
#         TABLE: BigQuery table name

#     Returns: pandas DataFrame
#     '''

#     client = bigquery.Client(project=PROJECT)

#     query = f'''
#     SELECT *
#     FROM `{PROJECT}.{DATASET}.{TABLE}`
#     '''

#     df = client.query(query).to_dataframe()

#     rows, cols = df.shape
#     print(f'Loaded {rows:,} rows and {cols} columns from {PROJECT}.{DATASET}.{TABLE}')

#     return df

In [ ]:
df = load_from_bigquery('gridzero-489711', 'merged_set', 'feature_engineered_dataset_2025')

In [ ]:
df.head(1)
df.dtypes

In [ ]:
def baseline_model_xgb(PROJECT: str='gridzero-489711',
                       DATASET: str='merged_set',
                       TABLE: str='feature_engineered_dataset_2025',
                       test_size:int = 0.3):

    # pull df from BQ
    df = load_from_bigquery(PROJECT=PROJECT, DATASET=DATASET, TABLE=TABLE)

    # set features and target variables
    X = df.drop(columns=['carbon_intensity'])
    y = df['carbon_intensity']

    # test train split for single year
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)

    #put column transformer into pipeline to ignore the columns
    cols_to_keep_X = [c for c in df.columns if c not in ['datetime']]

    col_tf = ColumnTransformer(transformers=[
                ('choose_cols', 'passthrough', cols_to_keep_X)
        ])

    # build simple xgb model
    pipeline = make_pipeline(
                    col_tf,
                    StandardScaler(),
                    XGBRegressor()
                 )

    # training model
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test.iloc[:5])
    print(y_pred)
    return pipeline

test_model = baseline_model_xgb()
test_model


In [ ]:
test_input = df.iloc[:3]
# test_input = test_input.drop(columns=['datetime', 'carbon_intensity'])
# print(test_input)
model.predict(test_input)


In [ ]:
print(f'Shape: {df.shape}')
print(f'Nulls:\n{df.isnull().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')

In [ ]:
df['Fossil Gas']

In [ ]:
cols = [df.columns]
cols

In [ ]:

df.plot(x='time', y='carbon_intensity', figsize=(15,4))
plt.title('Carbon Intensity 2025')
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm')
plt.title('Feature Correlations')
plt.show()

In [ ]:
feat_corr_matrix_plot, ax = plt.subplots()

corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', ax = ax)
plt.title('Feature Correlations')
plt.show()

In [ ]:
feat_corr_matrix_plot

In [ ]:
load_from_bigquery('gridzero-489711', 'merged_set', 'Fully_merged_dataset_2025')

In [ ]:
df.rename(columns={'time': 'datetime'})
df = df.rename(columns={'time': 'datetime'})

In [ ]:
# import pandas as pd

# def engineer_features(df, target_col='carbon_intensity', add_year_lag=False):
#     '''
#     Create lag and calendar features for modelling.

#     Parameters
#     ----------
#     df : pd.DataFrame
#         Input dataframe containing a datetime column and target column.

#     target_col : str, default='carbon_intensity'
#         Name of the target column to create lag features from.

#     add_year_lag : bool, default=False
#         Whether to add a 1-year lag feature.
#         Only use this if the dataframe contains at least 2 years of half-hourly data.

#     Returns
#     -------
#     pd.DataFrame
#         DataFrame with engineered lag and calendar features.
#     '''

#     df = df.copy()

#     # Ensure datetime is in datetime format
#     df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

#     # Drop rows with invalid datetime values
#     df = df.dropna(subset=['datetime'])

#     # Sort by datetime
#     df = df.sort_values('datetime').reset_index(drop=True)

#     # Lag features
#     # 48 half-hour periods = 24 hours
#     df['lag_48'] = df[target_col].shift(48)

#     # # 336 half-hour periods = 7 days
#     df['lag_336'] = df[target_col].shift(336)

#     # # Optional yearly lag
#     # # 17520 half-hour periods = 365 days
#     if add_year_lag:
#         df['lag_17520'] = df[target_col].shift(17520)

#     # Calendar features
#     df['hour'] = df['datetime'].dt.hour
#     df['day_of_week'] = df['datetime'].dt.dayofweek
#     df['month'] = df['datetime'].dt.month
#     df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

#     return df


# def validate_features(df):
#     '''
#     Print validation checks for the engineered dataset.

#     Parameters
#     ----------
#     df : pd.DataFrame
#         Feature-engineered dataframe.

#     Returns
#     -------
#     None
#     '''

#     print('Shape:', df.shape)

#     print('\nColumns:')
#     print(df.columns.tolist())

#     print('\nMissing values:')
#     print(df.isnull().sum())

#     print('\nDuplicate rows:')
#     print(df.duplicated().sum())

#     print('\nDuplicate datetimes:')
#     if 'datetime' in df.columns:
#         print(df['datetime'].duplicated().sum())
#     else:
#         print('datetime column not found')

#     print('\nDatetime spacing:')
#     if 'datetime' in df.columns:
#         print(df['datetime'].diff().value_counts().head())
#     else:
#         print('datetime column not found')


# def drop_lag_nulls(df):
#     '''
#     Drop rows with null values created by lag features.

#     Parameters
#     ----------
#     df : pd.DataFrame
#         Feature-engineered dataframe.

#     Returns
#     -------
#     pd.DataFrame
#         DataFrame with null rows removed and index reset.
#     '''

#     return df.dropna().reset_index(drop=True)


In [ ]:
display(df.head(5))

display(engineer_features(df, target_col='carbon_intensity'))

In [ ]:
df = engineer_features(df)
validate_features(df)
df = drop_lag_nulls(df)
df

In [ ]:
# data features and target
# note dropped target AND datime
# note renamed time to datetime to make sure engineerign features work
X = df.drop(columns=['carbon_intensity', 'datetime'])
y = df['carbon_intensity']

print(X.dtypes)

In [ ]:
y


In [ ]:
# test train split for 2025
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.30)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# test train split for more years
# df = df.sort_values('datetime').copy()

# split_date = '2025-10-01'

# train_df = df[df['datetime'] < split_date]
# test_df = df[df['datetime'] >= split_date]

# X_train = train_df.drop(columns=['carbon_intensity', 'datetime'])
# y_train = train_df['carbon_intensity']

# X_test = test_df.drop(columns=['carbon_intensity', 'datetime'])
# y_test = test_df['carbon_intensity']

In [ ]:
# simple pipelines

# pipeline = make_pipeline(
#     XGBRegressor(
#         n_estimators=100,
#         learning_rate=0.1,
#         max_depth=6,
#     )
# )

# pipeline_xgbr1 = make_pipeline(
#     StandardScaler(),
#     XGBRegressor(
#         n_estimators=100,
#         learning_rate=0.1,
#         max_depth=6,
#         random_state=42
#     )
# )


In [ ]:
# pipeline.fit(X_train, y_train)
# y_pred = pipeline.predict(X_test)
# print(y)
# print(y_pred)

In [ ]:
# results = cross_validate(
#     pipeline,
#     X_train,
#     y_train,
#     cv=5,
#     scoring=['neg_mean_squared_error', 'neg_root_mean_squared_error', 'r2', 'neg_mean_absolute_error', 'neg_max_error']
# )

# results


In [ ]:
# test_results = {k: v for k, v in results.items() if k.startswith('test')}
# test_results = pd.DataFrame(test_results)

# display(test_results)

In [ ]:
# plt.figure(figsize=(10,4))
# plt.plot(y_test.values, label='true')
# plt.plot(y_pred, label='pred')
# plt.show()

In [ ]:
# function to extract from datetime
# def extract_datetime_features(df):
#     df = df.copy()
#     dt = pd.to_datetime(df.iloc[:, 0])

#     return pd.DataFrame({
#         'hour': dt.dt.hour,
#         'day': dt.dt.day,
#         'month': dt.dt.month,
#         'dayofweek': dt.dt.dayofweek
#     }, index=df.index)

In [ ]:
# model comparison
models = {
    'dummy': make_pipeline(
        DummyRegressor(strategy='mean')
    ),
    'linear_regression': make_pipeline(
        StandardScaler(),
        LinearRegression()
    ),
    'random_forest': make_pipeline(
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )
    ),
    'xgboost_default': make_pipeline(
            XGBRegressor()
    ),

    'xgboost': make_pipeline(
        XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=6,
            random_state=42
        )
    ),
    'xgboost_scl': make_pipeline(
        StandardScaler(),
        XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=6,
            random_state=42
        )
    ),
    'xgboost_opt' : make_pipeline(
        XGBRegressor(
            n_estimators=2000,
            learning_rate=0.03,
            max_depth=5,
            #L1
            reg_alpha=0.1,
            #L2
            reg_lambda=0,
            # histo method (faster than default greedy algo)
            tree_method='hist',
            random_state=42
            )
    ),

    'xgboost_opt_scl' : make_pipeline(
        StandardScaler(),
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=5,
            reg_alpha=0.1,
            reg_lambda=1.0,
            tree_method='hist',
            random_state=42
            )

    )
}

In [ ]:
# scoring

scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error',
    'r2': 'r2',
    'max_err':'neg_max_error'
}

In [ ]:
# loop through models

# results
results = []

for model_name, pipeline in models.items():
    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        return_train_score=False
    )

    model_results = {
        'model': model_name,
        'model_fit_t': cv_results['fit_time'].mean(),
        'mae_mean': -cv_results['test_mae'].mean(),
        'rmse_mean': -cv_results['test_rmse'].mean(),
        'r2_mean': cv_results['test_r2'].mean(),
        'max_err_max': -cv_results['test_max_err'].max(),
    }

    results.append(model_results)

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('mae_mean')

display(results_df)

In [ ]:
# results:
# XGBOOST low MAE/RMSE/r2

In [ ]:
# simple model and predict
# pipeline = pipeline = make_pipeline(
#         StandardScaler(),
#         XGBRegressor()
# )

# pipeline.fit(X_train, y_train)

# y_pred = pipeline.predict(X_test)
# y_pred


In [ ]:
# pipeline to use

# pipeline = pipeline = make_pipeline(
#         StandardScaler(),
#         XGBRegressor(
#             n_estimators=2000,
#             learning_rate=0.03,
#             max_depth=5,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             reg_alpha=0.1,
#             reg_lambda=1.0,
#             #histo
#             tree_method='hist',
#             random_state=42
#             )
#         )

In [ ]:
# gridsearch

# params = {
#             'n_estimators': np.arange(200, 1000, 100),
#             'max_depth': np.arange(3, 11, 1),
#             'learning_rate': np.arange(0.01, 0.3, 0.01),
#             'subsample': [0.8, 1.0],
#             'colsample_bytree': [0.8, 1.0]
# }


In [ ]:
# grid = GridSearchCV(
#     estimator=xgb,
#     param_grid=param_grid,
#     scoring='neg_mean_absolute_error',
#     cv=tscv,
#     n_jobs=-1,
#     verbose=2
# )

# grid.fit(X_train, y_train)

In [ ]:
# # best model
# best_model = grid.best_estimator_

In [ ]:
# predicting
# y_pred = pipeline.predict(X_test)

In [ ]:
# fit with girdsearch cv


In [ ]:
baseline_model_xgb()